
# Introduction

[Scrapy](https://scrapy.org/) est un framework permettant de crawler des
sites web et d'en extraire les données de façon structurée.

## Installation

[Scrapy](https://scrapy.org/) ne fait pas partie de la distribution par
défaut de Python et doit être installé manuellement. Ici, le package est
déjà installé grâce à Pipenv.

Si vous avez besoin de l'installer dans un autre cadre.

-   Avec **Pipenv** : `pipenv install scrapy==2.11.2`

**Vérifiez que vous avez bien activé votre environnement virtuel Python !!**

In [1]:
!which python

/opt/venv/bin/python


In [2]:
import scrapy

## Architecture

[Scrapy](https://scrapy.org/) est un framework comportant plusieurs
composants.

<img src="images/architecture.png" alt="image" class="align-center" />

L'ensemble du processus est contrôlé par l'**engine** (les termes anglo
saxons ont été retenus pour un meilleur référencement dans la
[documentation officielle](https://docs.scrapy.org/en/latest/)).

Le framework est articulé avec plusieurs composants qui gèrent chacun un
rôle différent. Nous allons les détailler.

-   Les **Spiders** : permettent de naviguer sur un site et de
    référencer les règles d'extraction de la donnée.
-   Les **Pipelines** : font le lien entre la donnée brute et des objets
    structurés
-   Les **Middlewares** : permettent d'effectuer des transformations sur
    les objets ou sur les requêtes exécutées par l'engine.
-   Le **Scheduler** : gère l'ordre et le timing des requêtes
    effectuées.

## Fonctionnement

[Scrapy](https://scrapy.org/) est entièrement organisé autour d'un
composant central : l'*engine*.

Le rôle de l'*engine* est de contrôler le flux de données entre les
différents composants du système.

1.  En particulier, il est chargé de récupérer les *requests* définies
    dans les *spiders*
2.  Ces *requests* sont ensuite fournies au *scheduler* qui se charge de
    leur ordonnancement
3.  Les *requests* sont présentées selon cet ordonnancement à
    l'*engine*...
4.  ... qui les transmet au *downloader*
5.  Le *downloader* effectue la *request* et transmet la *response* (le
    contenu de la page web) à l'*engine*...
6.  ... puis l'envoie au *spider* pour traitement
7.  Le *spider* génére des *items* qui sont transmis à l'*engine*
8.  Les *items* sont ensuite poussés dans un pipeline pour nettoyage,
    validation et stockage

Ce processus est répété jusqu'à épuisement des requêtes.

[Scrapy](https://scrapy.org/) est un [framework orienté
événements](https://en.wikipedia.org/wiki/Event-driven_architecture)
(basé sur [Twisted](https://twistedmatrix.com/)) permettant une
programmation asynchrone (non bloquante). C'est particulièrement
intéressant dans les opérations de scraping, puisque **le programme
n'attend pas le résultat d'une requête pour en lancer une autre**.

En effet, lorsque l'on sollicite une ressource (requête réseau, système
de fichier, etc.) en mode bloquant, l'exécution du programme est
suspendue le temps que la transaction avec la ressource se termine (par
exemple le temps qu'une page web soit complètement téléchargée).
L'intérêt de faire des appels non bloquants, c'est que l'on peut gérer
de multiples téléchargements en parallèle, et que le programme peut
continuer à tourner pendant ce temps.

# Un scraping élémentaire

Avant de rentrer dans les détails du framework, nous allons mettre en
oeuvre un premier script permettant de récupérer l'information présente
sur [la page web](http://evene.lefigaro.fr/citations/winston-churchill)
recensant les citations de [Sir Winston
Churchill](https://en.wikipedia.org/wiki/Winston_Churchill).

**Exercice**

Examiner le code source de cette page avec l'inspecteur de votre
navigateur. Identifier les éléments contenant l'information recherchée,
ici la chaîne de caractères contenant la citation proprement dite.

## Le code source

Le code utilisé est le suivant:

> **Réponse (examiner le code source)** — La page a été refaite depuis l'écriture du cours. Avec l'inspecteur (clic droit → *Inspecter*), chaque citation est maintenant dans :
> ```html
> <article class="fig-quote">
>   <blockquote class="fig-quote__content">
>     <p class="fig-quote__text"><a href="...">Texte de la citation</a></p>
>   </blockquote>
>   <p class="fig-credit">Par <a class="fig-link fig-credit__author">Winston Churchill</a></p>
> </article>
> ```
> L'ancien sélecteur `//div[@class="figsco__quote__text"]` ne trouve donc plus rien : le spider ci-dessous utilise `//article[@class="fig-quote"]`, puis `.//p[@class="fig-quote__text"]/a/text()` pour le texte.

*Version mise à jour du script (nouveaux sélecteurs + réponses aux exercices plus bas : guillemets retirés, auteur, filtre Churchill) :*

In [3]:
# %load citations_churchill_spider1.py
import scrapy


class ChurchillQuotesSpider(scrapy.Spider):
    name = "citations de Churchill"
    start_urls = ["https://evene.lefigaro.fr/citations/winston-churchill", ]
    # Pour que les accents restent lisibles dans le fichier JSON exporté
    custom_settings = {"FEED_EXPORT_ENCODING": "utf-8", "REQUEST_FINGERPRINTER_IMPLEMENTATION": "2.7"}

    def parse(self, response):
        # Le site a été refait depuis l'écriture du cours : chaque citation est
        # maintenant dans <article class="fig-quote">, le texte dans
        # <p class="fig-quote__text"><a>...</a></p> et l'auteur dans
        # <a class="fig-link fig-credit__author">.
        for cit in response.xpath('//article[@class="fig-quote"]'):
            text_value = cit.xpath('.//p[@class="fig-quote__text"]/a/text()').extract_first()
            author = cit.xpath('.//a[contains(@class, "fig-credit__author")]/text()').extract_first()

            text_value = self.clean_quote(text_value)
            author = author.strip() if author else None

            # On ne garde que les citations de Winston Churchill
            if text_value and author == "Winston Churchill":
                yield {"text": text_value, "author": author}

    def clean_quote(self, text):
        """Retire les guillemets “ (U+201C) et ” (U+201D) et les espaces superflus."""
        if text is None:
            return None
        text = text.replace("\u201c", "").replace("\u201d", "")
        return " ".join(text.split())

## Le fonctionnement

Le fonctionnement est le suivant:

-   On importe le module scrapy (2)
-   et on définit une sous classe de `scrapy.Spider` (4)
-   la variable `start_urls` contient la liste des pages à scraper (6)
-   On redéfinit la méthode $parse$ dont la signature est définie dans
    la classe mère (8)
-   L'objet
    [response](https://docs.scrapy.org/en/latest/topics/request-response.html#response-objects)
    représente la réponse à la requête HTTP (l'attribut $text$ permet
    d'accéder à son contenu). On recherche ensuite tous les containers
    `<div>` identifiés dans l'exercice précédent. Ici la page est
    particulièrement bien structurée et les citations disposent de leur
    propre container, identifié par l'attribut `class` de valeur
    `figsco__quote__text`. La sélection se fait par une expression
    [XPath](https://en.wikipedia.org/wiki/XPath), un langage de
    sélection de noeud dans un document XML (9). En langage naturel, la
    requête pourrait se formuler : "On recherche tous les containers
    `<div>` dont la valeur de l'attribut `class` est égal à
    `figsco__quote__text`".
-   Pour chaque résultat, on construit un dictionnaire dont la clé est
    `text` et la valeur le contenu du lien `<a>`. Ce résultat est fourni
    par un générateur ($yield$) (11).

On lance le scraping depuis un terminal (attention, ici on ne lance pas la cellule du jupyter notebook mais vraiment le script contenu dans le ifchier citations_churchill_spider1.py):

In [4]:
!scrapy runspider citations_churchill_spider1.py -O data/citation.json

2026-11-24 13:10:04 [scrapy.utils.log] INFO: Scrapy 2.11.2 started (bot: scrapybot)


2026-11-24 13:10:04 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.12.9, cssselect 1.3.0, parsel 1.10.0, w3lib 2.3.1, Twisted 25.5.0, Python 3.12.16 (main, Nov  5 2026, 08:06:14) [GCC 14.2.0], pyOpenSSL 25.3.0 (OpenSSL 3.5.4 30 Sep 2025), cryptography 46.0.3, Platform Linux-7.0.14-linuxkit-aarch64-with-glibc2.41
2026-11-24 13:10:04 [scrapy.addons] INFO: Enabled addons:
[]
2026-11-24 13:10:04 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.epollreactor.EPollReactor
2026-11-24 13:10:04 [scrapy.extensions.telnet] INFO: Telnet Password: e9ec6e06bd4dc4c7
2026-11-24 13:10:04 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-11-24 13:10:04 [scrapy.crawler] INFO: Overridden settings:
{'FEED_EXPORT_ENCODING': 'utf-8',
 'REQUEST_FINGERPRINTER_IMPLEMENTATION': '2.

2026-11-24 13:10:04 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://evene.lefigaro.fr/citations/winston-churchill> (referer: None)


2026-11-24 13:10:04 [scrapy.core.scraper] DEBUG: Scraped from <200 https://evene.lefigaro.fr/citations/winston-churchill>
{'text': "La Russie est un rébus enveloppé de mystère au sein d'une énigme.", 'author': 'Winston Churchill'}
2026-11-24 13:10:04 [scrapy.core.scraper] DEBUG: Scraped from <200 https://evene.lefigaro.fr/citations/winston-churchill>
{'text': "Je n'ai rien à offrir que du sang, du labeur, des larmes et de la sueur.", 'author': 'Winston Churchill'}
2026-11-24 13:10:04 [scrapy.core.scraper] DEBUG: Scraped from <200 https://evene.lefigaro.fr/citations/winston-churchill>
{'text': 'La démocratie est un mauvais système, mais elle est le moins mauvais de tous les systèmes.', 'author': 'Winston Churchill'}
2026-11-24 13:10:04 [scrapy.core.scraper] DEBUG: Scraped from <200 https://evene.lefigaro.fr/citations/winston-churchill>
{'text': 'Le golf consiste à mettre une balle de 4 cm de diamètre sur une boule de 40.000 km de tour et à frapper la petite, non la grande.', 'author': '

On y trouve des informations sur les paramètres
utilisés:

```
2018-01-10 17:21:05 [scrapy.utils.log] INFO: Scrapy 1.4.0 started (bot: scrapybot)
2018-01-10 17:21:05 [scrapy.utils.log] INFO: Overridden settings: {'SPIDER_LOADER_WARN_ONLY': True}
```

les
[extensions](https://docs.scrapy.org/en/latest/topics/extensions.html)
...:

```
2018-01-10 17:21:05 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
'scrapy.extensions.telnet.TelnetConsole',
'scrapy.extensions.logstats.LogStats']
```

[Les composants middleware
downloader](https://docs.scrapy.org/en/latest/topics/downloader-middleware.html)
... :

```
2018-01-10 17:21:05 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
'scrapy.downloadermiddlewares.defaultheaders.DefaultHeadersMiddleware',
'scrapy.downloadermiddlewares.useragent.UserAgentMiddleware',
'scrapy.downloadermiddlewares.retry.RetryMiddleware',
'scrapy.downloadermiddlewares.redirect.MetaRefreshMiddleware',
'scrapy.downloadermiddlewares.httpcompression.HttpCompressionMiddleware',
'scrapy.downloadermiddlewares.redirect.RedirectMiddleware',
'scrapy.downloadermiddlewares.cookies.CookiesMiddleware',
'scrapy.downloadermiddlewares.httpproxy.HttpProxyMiddleware',
'scrapy.downloadermiddlewares.stats.DownloaderStats']
```

Idem pour [les composants middleware
spider](https://docs.scrapy.org/en/latest/topics/spider-middleware.html)
...:

```
2018-01-10 17:21:05 [scrapy.middleware] INFO: Enabled spider middlewares:
['scrapy.spidermiddlewares.httperror.HttpErrorMiddleware',
'scrapy.spidermiddlewares.offsite.OffsiteMiddleware',
'scrapy.spidermiddlewares.referer.RefererMiddleware',
'scrapy.spidermiddlewares.urllength.UrlLengthMiddleware',
'scrapy.spidermiddlewares.depth.DepthMiddleware']
```

Aucun
[pipeline](https://docs.scrapy.org/en/latest/topics/item-pipeline.html)
n'est activé :

```
2018-01-10 17:21:05 [scrapy.middleware] INFO: Enabled item pipelines:
[]
```

**Exercice**

Identifier la position des [composants middleware
downloader](https://docs.scrapy.org/en/latest/topics/downloader-middleware.html),
des [composants middleware
spider](https://docs.scrapy.org/en/latest/topics/spider-middleware.html)
et du
[pipeline](https://docs.scrapy.org/en/latest/topics/item-pipeline.html)
dans $l'architecture <Introduction>$

L'exécution du scraping proprement dit débute :
```
2018-01-10 17:21:05 [scrapy.core.engine] INFO: Spider opened
2018-01-10 17:21:05 [scrapy.extensions.logstats] INFO: Crawled 0 pages (at 0 pages/min), scraped 0 items (at 0 items/min)
2018-01-10 17:21:05 [scrapy.extensions.telnet] DEBUG: Telnet console listening on 127.0.0.1:6023
```

> **Réponse (position des composants dans l'architecture)**
> - Les **downloader middlewares** sont placés **entre l'engine et le downloader** : chaque requête les traverse avant d'être envoyée sur le web, et chaque réponse les traverse au retour (gestion du User-Agent, des retries, des redirections, des cookies, de la compression...).
> - Les **spider middlewares** sont placés **entre l'engine et les spiders** : ils voient passer les réponses envoyées aux spiders et ce que les spiders renvoient (items et nouvelles requêtes). Ex : `OffsiteMiddleware` filtre les requêtes hors de `allowed_domains`, `DepthMiddleware` limite la profondeur.
> - Les **item pipelines** sont **en sortie** : les items produits par les spiders passent par l'engine puis par les pipelines (nettoyage, validation, stockage). Ici la liste est vide car un `runspider` simple n'en active aucun.

La première URL est poussée par le scheduler:
```
2018-01-10 17:21:05 [scrapy.core.engine] DEBUG: Crawled (200) <GET http://evene.lefigaro.fr/citations/winston-churchill> (referer: None)
```

## Les résultats

Les résultats sont fournis par le générateur défini dans la méthode
$parse$ dans un dictionnaire. Ils contiennent le texte des citations
dans la valeur de la clé `text` :
```
2018-01-10 17:21:05 [scrapy.core.scraper] DEBUG: Scraped from <200 http://evene.lefigaro.fr/citations/winston-churchill>
{'text': '“Le vice inhérent au capitalisme consiste en une répartition inégale des richesses. La vertu inhérente au socialisme consiste en une égale répartition de la misère.”'}
2018-01-10 17:21:05 [scrapy.core.scraper] DEBUG: Scraped from <200 http://evene.lefigaro.fr/citations/winston-churchill>
{'text': "Faire le bien, éviter le mal, c'est ça le paradis."}
```

## Les statistiques

Une fois le scraping effectué, quelques statistiques sont affichées sur
le terminal:
```
2018-01-10 17:21:05 [scrapy.core.engine] INFO: Closing spider (finished)
2018-01-10 17:21:05 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 242,
'downloader/request_count': 1,
'downloader/request_method_count/GET': 1,
'downloader/response_bytes': 17435,
'downloader/response_count': 1,
'downloader/response_status_count/200': 1,
'finish_reason': 'finished',
'finish_time': datetime.datetime(2018, 1, 10, 16, 21, 5, 858347),
'item_scraped_count': 16,
'log_count/DEBUG': 18,
'log_count/INFO': 7,
'response_received_count': 1,
'scheduler/dequeued': 1,
'scheduler/dequeued/memory': 1,
'scheduler/enqueued': 1,
'scheduler/enqueued/memory': 1,
'start_time': datetime.datetime(2018, 1, 10, 16, 21, 5, 645347)}
2018-01-10 17:21:05 [scrapy.core.engine] INFO: Spider closed (finished)
```

On observe notamment que notre code permet de récupérer la taille de la
page web (17435 bytes), le temps d'exécution à partir des valeurs
`finish_time` et `start_time`, le nombre d'items scrapés (16), etc...

**Exercice**

Les citations extraites sont elles toutes de [Sir Winston
Churchill](https://en.wikipedia.org/wiki/Winston_Churchill) ? Il sera
peut être nécessaire de modifier le sélecteur XPath. Nous verrons ça
lorsque il faudra récupérer les données relative à l'auteur.


> **Réponse** — On compte les auteurs de **toutes** les citations de la page, sans filtre :

In [5]:
import requests
from collections import Counter
from scrapy.http import HtmlResponse

UA = {"User-Agent": "Scrapy/2.11.2 (+https://scrapy.org)"}
r = requests.get("https://evene.lefigaro.fr/citations/winston-churchill", headers=UA, timeout=10)
response = HtmlResponse(url=r.url, body=r.content, encoding="utf-8")  # même objet que dans scrapy shell

auteurs = Counter(
    a.strip() for a in response.xpath('//article[@class="fig-quote"]//a[contains(@class, "fig-credit__author")]/text()').extract()
)
print(len(response.xpath('//article[@class="fig-quote"]')), "citations sur la page")
auteurs

69 citations sur la page


Counter({'Winston Churchill': 69})

> Sur la page actuelle, toutes les citations sont bien attribuées à Winston Churchill. Avec l'ancien sélecteur, ce n'était pas garanti (le premier résultat de `data/citation.json` d'origine était même vide : `{"text": null}`). Le spider vérifie donc explicitement l'auteur (`if author == "Winston Churchill"`) : si la page affiche un jour des citations d'autres personnes, elles seront ignorées.

## Modifier les données

Il est parfois nécessaire de faire un traitement sur les données
scrapées, pour ajouter ou retirer de l'information.

**Exercice**

Retirer les caractères `“` et `”` qui délimitent la citation. Ces
caractères sont identifiés en Unicode comme [LEFT DOUBLE QUOTATION
MARK](http://www.fileformat.info/info/unicode/char/201c/index.htm) et
[RIGHT DOUBLE QUOTATION
MARK](http://www.fileformat.info/info/unicode/char/201d/index.htm).

Attention, ici il faut bien modifier directement le script `citations_churchill_spider1.py` et lancer dans une cellule ou votre terminal la commande utilisée précédemment : `scrapy runspider citations_churchill_spider1.py`

> **Réponse** — Dans le script `citations_churchill_spider1.py`, la méthode `clean_quote()` retire les deux caractères Unicode avec `text.replace("\u201c", "").replace("\u201d", "")` (U+201C = `“`, U+201D = `”`), puis supprime les espaces superflus avec `" ".join(text.split())`.

## Plus de données

Il est souvent nécessaire de récupérer plusieurs informations relatives
à un même item. Dans cet exemple, il est judicieux d'associer à la
citation le nom de son auteur, en allant chercher cette information au
plus près du texte lui même.

**Exercice**

Examiner le code source de la page web et identifier la structuration de
la donnée associée à l'auteur. En déduire l'expression XPath permettant
de la récupérer. S'assurer que seules les citations de [Sir Winston
Churchill](https://en.wikipedia.org/wiki/Winston_Churchill) sont
extraites. Ajouter une clé `author` au dictionnaire retourné par le
$yield$ dont la valeur est précisément la chaîne de caractères contenant
l'auteur.

Un exemple de dictionnaire retourné:
```
{
    'text': "Si deux hommes ont toujours la même opinion, l'un d'eux est de trop.", 
    'author': 'Winston Churchill'
}
```

> **Réponse** — L'auteur est dans le lien `<a class="fig-link fig-credit__author">` de chaque `<article class="fig-quote">`. L'expression XPath (relative à la citation) est :
> ```
> .//a[contains(@class, "fig-credit__author")]/text()
> ```
> On utilise `contains()` car l'attribut `class` contient plusieurs classes (`fig-link fig-credit__author`). Le spider renvoie `{"text": ..., "author": ...}` et ne garde que les citations dont l'auteur est `Winston Churchill`. Résultat de la dernière exécution (fichier `data/citation.json`) :

In [6]:
import pandas as pd

citations = pd.read_json("data/citation.json")
print(len(citations), "citations")
citations.head(10)

69 citations


,text,author
0,La Russie est un rébus enveloppé de mystère au...,Winston Churchill
1,"Je n'ai rien à offrir que du sang, du labeur, ...",Winston Churchill
2,"La démocratie est un mauvais système, mais ell...",Winston Churchill
3,Le golf consiste à mettre une balle de 4 cm de...,Winston Churchill
4,"Une pomme par jour éloigne le médecin, pourvu ...",Winston Churchill
5,"Il n'y a qu'une réponse à la défaite, et c'est...",Winston Churchill
6,"Lorsque je suis à l'étranger, je m'impose de n...",Winston Churchill
7,Je ne contredis pas Monsieur le député lorsqu’...,Winston Churchill
8,Certains hommes changent de parti en fonction ...,Winston Churchill
9,"Avoir foi en la perfection de l’homme, c’est t...",Winston Churchill


si vous voulez stocker les données scrapées dans un fichier vous pouvez grâce à :
```
scrapy runspider citations_churchill_spider1.py -o data/citation.json -t json
```


# Votre premier projet


Dans un premier temps vous devez créer un projet Scrapy avec la commande
:

```
scrapy startproject monprojet
```
*(Commande non exécutée ici : le dépôt contient déjà le projet `newscrawler` créé avec cette commande, et on travaille dedans.)*

Cette commande va créer un dossier `monprojet` contenant les éléments
suivants correspondant au squelette :

```
newscrawler/
    scrapy.cfg            # Options de déploiement

    newscrawler/             # Le module Python contenant les informations
        __init__.py

        items.py          # Fichier contenant les items

        middlewares.py    # Fichier contenant les middlewares

        pipelines.py      # Fichier contenant les pipelines

        settings.py       # Fichier contenant les paramètres du projet

        spiders/          # Dossier contenant toutes les spiders
            __init__.py
```

**Attention, à partir de maintenant vous créerez et modifierez les fichiers python dans votre projet scrapy et non pas dans les cellules jupyter qui sont là juste pour l'exemple**

Pour la suite, un projet scrapy du nom de `newscrawler` est présent dans le repo pour vous guider, vous pouvez vous en inspirer pour implémenter vos propres spiders mais n'hésitez pas à explorer les méthodes que scrapy offre depuis la documentation.

# Votre première Spider

Une Spider est une classe Scrapy qui permet de mettre en place toute
l'architecture complexe vue dans l'introduction. Pour définir une
spider, il vous faut hériter de la classe $scrapy.Spider$. La seule
chose à faire est de définir la première requête à effectuer et comment
suivre les liens. La Spider s'arrêtera lorsqu'elle aura parcouru tous
les liens qu'on lui a demandé de suivre.

Pour créer une Spider on utilise la syntaxe:

```
scrapy genspider <SPIDER_NAME> <DOMAIN_NAME>
```

Par exemple,

> **Adaptation : Le Monde → franceinfo.** Depuis l'écriture du cours, Le Monde bloque Scrapy (erreur **406**) et refuse ses pages de rubriques aux robots (erreur **402** : « Votre trafic a été identifié comme automatisé »). On respecte ce blocage et on applique exactement la même démarche à **franceinfo.fr**, qui autorise les robots (voir son `robots.txt`). Les spiders `lemonde*` du cours sont conservés à titre d'exemple ; les versions qui fonctionnent sont `franceinfo_v2.py`, `franceinfo_v3.py` et `franceinfo_v4.py` dans `newscrawler/newscrawler/spiders/`.

In [7]:
!cd newscrawler && scrapy genspider franceinfo www.franceinfo.fr

Created spider 'franceinfo' using template 'basic' in module:
  newscrawler.spiders.franceinfo


Cette commande permet de créer une spider appelée `lemonde` pour scraper
le domaine `lemonde.fr`. Cela crée le fichier Python
`spiders/lemonde.py` suivant :

In [8]:
# %load newscrawler/newscrawler/spiders/lemonde.py
import scrapy


class LemondeSpider(scrapy.Spider):
    name = 'lemonde'
    allowed_domains = ['lemonde.fr']
    start_urls = ['http://lemonde.fr/']

    def parse(self, response):
        pass


Le fichier généré pour franceinfo :

In [9]:
!cat newscrawler/newscrawler/spiders/franceinfo.py

import scrapy


class FranceinfoSpider(scrapy.Spider):
    name = "franceinfo"
    allowed_domains = ["www.franceinfo.fr"]
    start_urls = ["https://www.franceinfo.fr"]

    def parse(self, response):
        pass


Une bonne pratique pour commencer à développer une Spider est de passer
par l'interface Shell proposée par Scrapy. Elle permet de récupérer un
objet `Response` et de tester les méthodes de récupération des données.

### ATTENTION : Les commandes scrapy shell doivent être lancées dans un terminal et non dans des cellules jupyter

`$ scrapy shell 'http://lemonde.fr'`

Pour les utilisateurs de windows il vous faut mettre des doubles quotes
:
```
scrapy shell "http://lemonde.fr"
```

Scrapy lance un kernel Python

Grâce à cette interface, vous avez accès à plusieurs objets comme la
`Response`, la `Request`, la `Spider` par exemple. Vous pouvez aussi
exécuter `view(response)` pour afficher ce que Scrapy récupère dans un
navigateur.

Voici quelques exemples de commandes que vous pouvez faire dans le scrapy shell :
```
In [1]: response
Out[1]: <200 https://www.lemonde.fr/>

In [3]: request
Out[3]: <GET https://www.lemonde.fr/>

In [4]: type(request)
Out[4]: scrapy.http.request.Request

In [5]: spider
Out[5]: <LemondeSpider 'lemonde' at 0x1080fccc0>

In [6]: type(spider)
Out[6]: monprojet.spiders.lemonde.LeMondeSpider
```

Ici on voit que la Spider est une instance de LemondeSpider. Lorsqu'on
lance le $scrapy shell$ scrapy va chercher dans les spiders si une
correspond au lien passé en paramètre, si oui , il l'utilise sinon une
$DefaultSpider$ est instanciée.

## Vos premières requêtes

On peut commencer à regarder comment extraire les données de la page web
en utilisant le langage de requêtes proposé par Scrapy. Il existe deux
types de requêtes : les requêtes `css` et `xpath`. Les requêtes `xpath`
sont plus complexes mais plus puissantes que les requêtes `css`. Dans le
cadre de ce tutorial, nous allons uniquement aborder les requêtes `css`,
elles nous suffiront pour extraire les données dont nous avons besoin
(en interne, Scrapy transforme les requêtes `css`en requêtes `xpath`.

Que ce soit les requêtes `css` ou `xpath`, elles crééent des sélecteurs
de différents types. Quelques exemples :

Pour récupérer le titre d'une page :
```
In [1]: response.css('title')
Out[1]: [<Selector xpath='descendant-or-self::title' data='<title>Le Monde.fr - Actualités et Infos'>]
```

On récupère une liste de sélecteurs correspondant à la requête `css`
appelée. La requête précédente était unique, d'autres requêtes moins
restrictives permettent de récupérer plusieurs résultats. Par exemple
pour rechercher l'ensemble des liens présents sur la page, on va
rechercher les balises HTML `<a></a>`

```
In [5]: response.css("a")[0:10]
Out[5]:
[<Selector xpath='descendant-or-self::a' data='<a target="_blank" data-target="jelec-he'>,
<Selector xpath='descendant-or-self::a' data='<a href="/"> <div class="logo__lemonde l'>,
<Selector xpath='descendant-or-self::a' data='<a href="https://secure.lemonde.fr/sfuse'>,
<Selector xpath='descendant-or-self::a' data='<a href="https://abo.lemonde.fr/#xtor=CS'>,
<Selector xpath='descendant-or-self::a' data='<a href="/" class="Burger__right-arrow j'>,
<Selector xpath='descendant-or-self::a' data='<a href="/" class="Burger__right-arrow j'>,
<Selector xpath='descendant-or-self::a' data='<a href="#" class="js-dropdown Burger__r'>,
<Selector xpath='descendant-or-self::a' data='<a href="/mouvement-des-gilets-jaunes/" '>,
<Selector xpath='descendant-or-self::a' data='<a href="/carlos-ghosn/" data-suggestion'>,
<Selector xpath='descendant-or-self::a' data='<a href="/implant-files/" data-suggestio'>]
```

Pour récupérer le texte contenu dans les balises, on passe le paramètre
`<TAG>::text`. Par exemple :

```
In [6]: response.css("title::text")
Out[6]: [<Selector xpath='descendant-or-self::title/text()' data='Le Monde.fr - Actualités et Infos en Fra'>]
```

### Exercice

Comparer les résultats des deux requêtes `response.css('title')` et
`response.css('title::text')`.

Maintenant pour extraire les données des selecteurs on utilise l'une des
deux méthodes suivantes :
- `extract()` permet de récupérer une liste
des données extraites de tous les sélecteurs
- `extract_first()` permet
de récupérer une `String` provenant du premier sélecteur de la liste.


```
In [7]: response.css('title::text').extract_first()
Out[7]: 'Le Monde.fr - Actualités et Infos en France et dans le monde'
```

> **Réponse** — Pour faire les tests du `scrapy shell` dans le notebook, on construit le même objet `Response` à partir de la page d'accueil de franceinfo :

In [10]:
r = requests.get("https://www.franceinfo.fr/", headers=UA, timeout=10)
response = HtmlResponse(url=r.url, body=r.content, encoding="utf-8")

print(response.css("title"))
print(response.css("title::text"))
print(response.css("title::text").extract_first())

[<Selector query='descendant-or-self::title' data='<title>franceinfo - Actualités en tem...'>]
[<Selector query='descendant-or-self::title/text()' data='franceinfo - Actualités en temps réel...'>]
franceinfo - Actualités en temps réel et info en direct


> `response.css('title')` sélectionne **la balise entière** (`data='<title>franceinfo - ...</title>'`), alors que `response.css('title::text')` sélectionne **uniquement le texte** à l'intérieur de la balise. C'est la version `::text` qu'on utilise pour extraire une valeur.

On peut récupérer un attribut d'une balise avec la syntaxe
`<TAG>::attr(<ATTRIBUTE_NAME>)` :

Par exemple, les liens sont contenus dans un attribut `href`:

```
In [9]: response.css('a::attr(href)')[0:10]
Out[9]:
[<Selector xpath='descendant-or-self::a/@href' data='https://journal.lemonde.fr'>,
<Selector xpath='descendant-or-self::a/@href' data='/'>,
<Selector xpath='descendant-or-self::a/@href' data='https://secure.lemonde.fr/sfuser/connexi'>,
<Selector xpath='descendant-or-self::a/@href' data='https://abo.lemonde.fr/#xtor=CS1-454[CTA'>,
<Selector xpath='descendant-or-self::a/@href' data='/'>,
<Selector xpath='descendant-or-self::a/@href' data='/'>,
<Selector xpath='descendant-or-self::a/@href' data='#'>,
<Selector xpath='descendant-or-self::a/@href' data='/mouvement-des-gilets-jaunes/'>,
<Selector xpath='descendant-or-self::a/@href' data='/carlos-ghosn/'>,
<Selector xpath='descendant-or-self::a/@href' data='/implant-files/'>]
```

Comme vu précédemment, si on veut récupérer la liste des liens de la page on applique la méthode $extract()$

```
In [11]: response.css('a::attr(href)').extract()[0:10]
Out[11]:
['https://journal.lemonde.fr',
'/',
'https://secure.lemonde.fr/sfuser/connexion',
'https://abo.lemonde.fr/#xtor=CS1-454[CTA_LMFR]-[HEADER]-5-[Home]',
'/',
'/',
'#',
'/mouvement-des-gilets-jaunes/',
'/carlos-ghosn/',
'/implant-files/']
```

Les liens dans une page HTML sont souvent codés de manière relative par
rapport à la page courante. La méthode de l'objet `Response` peut être
utilisée pour recréer l'url complet.

Un exemple sur le 4e élément :
```
In [14]: response.urljoin(response.css('a::attr(href)').extract()[8])
Out[14]: 'https://www.lemonde.fr/carlos-ghosn/'
```

alors que :
```
In [15]: response.css('a::attr(href)').extract()[8]
Out[15]: '/carlos-ghosn/'
```

### Exercice : 

Utiliser une liste compréhension pour transformer les 10
premiers liens relatifs récupérés par la méthode `extract()` en liens
absolus.

Le résultat doit ressembler à :
```
Out[23]: 
['https://journal.lemonde.fr',
'https://www.lemonde.fr/',
'https://secure.lemonde.fr/sfuser/connexion',
'https://abo.lemonde.fr/#xtor=CS1-454[CTA_LMFR]-[HEADER]-5-[Home]',
'https://www.lemonde.fr/',
'https://www.lemonde.fr/',
'https://www.lemonde.fr/',
'https://www.lemonde.fr/mouvement-des-gilets-jaunes/',
'https://www.lemonde.fr/carlos-ghosn/',
'https://www.lemonde.fr/implant-files/']
```

> **Réponse** — `response.urljoin()` complète chaque lien relatif avec l'adresse de la page :

In [11]:
liens_relatifs = response.css("a::attr(href)").extract()[0:10]
liens_absolus = [response.urljoin(lien) for lien in liens_relatifs]

for avant, apres in zip(liens_relatifs, liens_absolus):
    print(f"{avant:<45} -> {apres}")

#base-page__main-content                      -> https://www.franceinfo.fr/#base-page__main-content
/                                             -> https://www.franceinfo.fr/
/en-direct/                                   -> https://www.franceinfo.fr/en-direct/
/videos-verticales/                           -> https://www.franceinfo.fr/videos-verticales/
/en-direct/radio.html                         -> https://www.franceinfo.fr/en-direct/radio.html
/en-direct/tv.html                            -> https://www.franceinfo.fr/en-direct/tv.html
https://jeux.franceinfo.fr/                   -> https://jeux.franceinfo.fr/
https://www.francetelevisions.fr/abonnements/information -> https://www.francetelevisions.fr/abonnements/information
https://meteo.franceinfo.fr/                  -> https://meteo.franceinfo.fr/
/societe/education/scolarite/                 -> https://www.franceinfo.fr/societe/education/scolarite/


## Des requêtes plus complexes

On peut créer des requêtes plus complexes en utilisant à la fois la
structuration HTML du document mais également la couche de présentation
CSS. On utilise l'inspecteur de `Google Chrome` pour identifier le type
et l'identifiant de la balise contenant les informations.

Il y a au moins deux choses à savoir en `css` :

-   Les `.` représentent les classes
-   Les `#` représentent les id

On se propose de récupérer toutes les sous-catégories de news dans la
catégorie **Actualités**. On remarque en utilisant l'inspecteur
d'élement de Chrome que toutes les catégories sont rangées dans une
balise avec l'id $#nav-markup$ ensuite dans les classes $Nav__item$.

A partir de cette structure HTML on peut construire la requête suivante
pour récupérer la barre de navigation:

```
In [19]: response.css("#nav-markup")
Out[19]: [<Selector xpath="descendant-or-self::*[@id = 'nav-markup']" data='<ul id="nav-markup"> <li class="Nav__ite'>]
```


Ensuite pour récupérer les différentes catégories :

```
In [24]: response.css('#nav-markup').css('.Nav__item')
Out[24]:
[<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item js-burger-to-show N'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item Nav__item--home Nav'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="/" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="/recherc'>]
```

On veut maintenant retourner tous les liens présents dans cette
catégorie. On remarque qu'elle apparait à la 4eme position.
```
In [34]: response.css("#nav-markup .Nav__item")[3]
Out[34]: <Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>
```

Maintenant pour récupérer tous les liens, on peut chainer les requêtes.
On accède alors à toutes les balises $a$.

```
In [35]: response.css("#nav-markup .Nav__item")[3].css("a")
Out[35]:
[<Selector xpath='descendant-or-self::a' data='<a href="#" class="js-dropdown Burger__r'>,
<Selector xpath='descendant-or-self::a' data='<a href="/mouvement-des-gilets-jaunes/" '>,
<Selector xpath='descendant-or-self::a' data='<a href="/carlos-ghosn/" data-suggestion'>,
<Selector xpath='descendant-or-self::a' data='<a href="/implant-files/" data-suggestio'>,
<Selector xpath='descendant-or-self::a' data='<a href="/climat/" data-suggestion>Clima'>,
<Selector xpath='descendant-or-self::a' data='<a href="/affaire-khashoggi/" data-sugge'>,
<Selector xpath='descendant-or-self::a' data='<a href="/emmanuel-macron/" data-suggest'>,
<Selector xpath='descendant-or-self::a' data='<a href="/ukraine/" data-suggestion>Ukra'>,
<Selector xpath='descendant-or-self::a' data='<a href="/russie/" data-suggestion>Russi'>,
<Selector xpath='descendant-or-self::a' data='<a href="/referendum-sur-le-brexit/" dat'>,
<Selector xpath='descendant-or-self::a' data='<a href="/harcelement-sexuel/" data-sugg'>,
<Selector xpath='descendant-or-self::a' data='<a href="/actualite-en-continu/" data-su'>,
<Selector xpath='descendant-or-self::a' data='<a href="/international/">International<'>,
<Selector xpath='descendant-or-self::a' data='<a href="/politique/">Politique</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/societe/">Société</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/les-decodeurs/">Les Décodeurs<'>,
<Selector xpath='descendant-or-self::a' data='<a href="/sport/">Sport</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/planete/">Planète</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/sciences/">Sciences</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/campus/">M Campus</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/afrique/">Le Monde Afrique</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/pixels/">Pixels</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/actualite-medias/">Médias</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/sante/">Santé</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/big-browser/">Big Browser</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/disparitions/">Disparitions</a'>]
```

Et pour récupérer les titres :
```
In [37]: response.css("#nav-markup .Nav__item")[3].css("a::text").extract()
Out[37]:
['Actualités',
 'Guerre au Proche-Orient',
 'Gouvernement Barnier',
 'Guerre en Ukraine',
 'Élection présidentielle américaine 2024',
 '7-Octobre, un an après',
 'Procès des viols de Mazan',
 'Donald Trump',
 'Climat',
 'Guadeloupe',
 'Toute l’actualité en continu',
 'International',
 'Politique',
 'Société',
 'Planète',
 'Climat',
 'Le Monde Afrique',
 'Les Décodeurs',
 'Sport',
 'Education',
 'M Campus',
 'Santé',
 'Intimités',
 'Sciences',
 'Pixels',
 'Disparitions',
 'Le Fil Good',
 'Podcasts',
 'Le Monde & Vous']
```

> **Adaptation franceinfo** — Sur franceinfo, les rubriques sont les titres de blocs de la section « Nos rubriques » de la page d'accueil : `<div id="rubriques">` puis les liens `<a class="title-link__link">`. Le texte est dans des sous-balises, d'où `*::text` :

In [12]:
rubriques = {
    " ".join(" ".join(lien.css("*::text").extract()).split()): response.urljoin(lien.attrib["href"])
    for lien in response.css("#rubriques a.title-link__link")
    if lien.attrib["href"] != "/une-information-transparente-franceinfo/"
}
rubriques

{'Sports': 'https://www.franceinfo.fr/sports/',
 'Culture': 'https://www.franceinfo.fr/culture/',
 'Vrai ou faux': 'https://www.franceinfo.fr/vrai-ou-fake/',
 'Politique': 'https://www.franceinfo.fr/politique/',
 'Société': 'https://www.franceinfo.fr/societe/',
 'Environnement': 'https://www.franceinfo.fr/environnement/',
 'Monde': 'https://www.franceinfo.fr/monde/'}

Le shell Scrapy permet de définir la structure des requêtes et de
s'assurer de la pertinence du résultat retourné. Pour automatiser le
processus, il faut intégrer cette syntaxe au code Python des modules de
spider définis dans la structure du projet.

## Intégration des requêtes

Le squelette de la classe `LeMondeSpider` généré lors de la création du
projet doit maintenant être enrichi. Par défaut 3 attributs et une
méthode `parse()` ont été créés :

-   `name` permet d'identifier sans ambiguïté la spider dans le code.
-   `allowed_domain` permet de filtrer les requêtes et forcer la spider
    à rester sur une liste de domaines.
-   `starts_urls` est la liste des urls d'où la spider va partir pour
    commencer son scraping.
-   `parse()` est une méthode héritée de la classe `scrapy.Spider`. Elle
    doit être redéfinie selon les requêtes que l'on doit effectuer et
    sera appelée sur l'ensemble des urls contenus dans la liste
    `starts_urls`.

`parse()` est une fonction `callback` qui sera appelée automatiquement
sur chaque objet `Response` retourné par la requête. Cette fonction est
appelée de manière asynchrone. Plusieurs requêtes peuvent ainsi être
lancées en parallèles sans bloquer le thread principal. L'objet
`Response` passé en paramètre est le même que celui mis à disposition
lors de l'exécution du Scrapy Shell.

In [13]:
def parse(self, response):
    title = response.css('title::text').extract_first()
    all_links = {
        name: response.urljoin(url) for name, url in zip(
            response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
            response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
    }
    yield {
        "title": title,
        "all_links": all_links
    }

La fonction est un générateur (`yield`) et retourne un dictionnaire
composé de deux éléments :

-   Le titre de la page;
-   La liste des liens sortants sous forme de String.

Pour le moment cette spider ne parcourt que la page d'accueil, ce qui
n'est pas très productif.

## Votre premier scraper

Récupérer les données sur un ensemble de pages webs nécessite d'explorer
en profondeur la structure du site en suivant tout ou partie des liens
rencontrés.

La spider peut se `balader` sur un site assez efficacement. Il suffit de
lui indiquer comment faire. Il faut spécifier à Scrapy de générer une
requête vers une nouvelle page en construisant l'objet `Request`
correspondant. Ce nouvel objet `Request` est alors inséré dans le
scheduler de Scrapy. On peut évidemment générer plusieurs `Request`
simultanément, correspondant par exemple, à différents liens sur la page
courante. Ils sont insérés séquentiellement dans le scheduler.

Pour cela, on modifie la méthode `parse()` de façon à ce qu'elle retourne
un objet `Request` pour chaque nouveau lien rencontré. On associe
également à cet objet une fonction de callback qui déterminera la
manière dont cette nouvelle page doit être extraite.

Par exemple, pour que la spider continue dans les liens des différentes
régions (pour l'instant la fonction de callback ne fait rien) :

In [14]:
# %load newscrawler/newscrawler/spiders/lemonde_v2.py
import scrapy
from scrapy import Request


class LemondeSpider(scrapy.Spider):
    name = "lemondev2"
    allowed_domains = ["www.lemonde.fr"]
    start_urls = ['https://www.lemonde.fr']

    def parse(self, response):
        title = response.css('title::text').extract_first()
        all_links = {
            name: response.urljoin(url) for name, url in zip(
                response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
                response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
        }
        yield {
            "title": title,
            "all_links": all_links
        }


Version franceinfo (`franceinfo_v2.py`) :

In [15]:
!cat newscrawler/newscrawler/spiders/franceinfo_v2.py

import scrapy
from scrapy import Request


class FranceinfoSpider(scrapy.Spider):
    """Équivalent de lemonde_v2.py pour franceinfo.fr.

    Le Monde bloque désormais Scrapy (erreur 406) et refuse ses pages de
    rubriques aux robots (erreur 402) : on applique la même démarche sur
    franceinfo, qui accepte les robots (voir https://www.franceinfo.fr/robots.txt).
    """
    name = "franceinfov2"
    allowed_domains = ["www.franceinfo.fr"]
    start_urls = ["https://www.franceinfo.fr/"]
    # Lien du bloc "rubriques" qui n'est pas une rubrique d'actualité
    excluded = ["/une-information-transparente-franceinfo/"]

    def parse(self, response):
        title = response.css("title::text").extract_first()
        # Les rubriques sont les titres de blocs de la section "Nos rubriques"
        # de la page d'accueil : <div id="rubriques"> ... <a class="title-link__link">
        all_links = {
            self.clean_spaces(" ".join(link.css("*::text").extract())): response.urljoin(link.a

Pour lancer la spider

In [16]:
!cd newscrawler && scrapy crawl franceinfov2 -L INFO -O ../data/franceinfo_v2.json && cat ../data/franceinfo_v2.json

2026-11-24 13:10:06 [scrapy.utils.log] INFO: Scrapy 2.11.2 started (bot: newscrawler)
2026-11-24 13:10:06 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.12.9, cssselect 1.3.0, parsel 1.10.0, w3lib 2.3.1, Twisted 25.5.0, Python 3.12.16 (main, Nov  5 2026, 08:06:14) [GCC 14.2.0], pyOpenSSL 25.3.0 (OpenSSL 3.5.4 30 Sep 2025), cryptography 46.0.3, Platform Linux-7.0.14-linuxkit-aarch64-with-glibc2.41
2026-11-24 13:10:06 [scrapy.addons] INFO: Enabled addons:
[]
2026-11-24 13:10:06 [scrapy.extensions.telnet] INFO: Telnet Password: 4d5994545747e153
2026-11-24 13:10:06 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-11-24 13:10:06 [scrapy.crawler] INFO: Overridden settings:
{'BOT_NAME': 'newscrawler',
 'DOWNLOAD_DELAY': 1,
 'FEED_EXPORT_ENCODING': 'utf-8',
 'LOG_LEVEL'

2026-11-24 13:10:06 [scrapy.middleware] INFO: Enabled spider middlewares:
['scrapy.spidermiddlewares.httperror.HttpErrorMiddleware',
 'scrapy.spidermiddlewares.referer.RefererMiddleware',
 'scrapy.spidermiddlewares.urllength.UrlLengthMiddleware',
 'scrapy.spidermiddlewares.depth.DepthMiddleware']
2026-11-24 13:10:06 [scrapy.middleware] INFO: Enabled item pipelines:
['newscrawler.pipelines.TextPipeline']
2026-11-24 13:10:06 [scrapy.core.engine] INFO: Spider opened
2026-11-24 13:10:06 [scrapy.extensions.logstats] INFO: Crawled 0 pages (at 0 pages/min), scraped 0 items (at 0 items/min)
2026-11-24 13:10:06 [scrapy.extensions.telnet] INFO: Telnet console listening on 127.0.0.1:6023


2026-11-24 13:10:07 [scrapy.core.engine] INFO: Closing spider (finished)
2026-11-24 13:10:07 [scrapy.extensions.feedexport] INFO: Stored json feed (1 items) in: ../data/franceinfo_v2.json
2026-11-24 13:10:07 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 446,
 'downloader/request_count': 2,
 'downloader/request_method_count/GET': 2,
 'downloader/response_bytes': 217882,
 'downloader/response_count': 2,
 'downloader/response_status_count/200': 2,
 'elapsed_time_seconds': 1.808804,
 'feedexport/success_count/FileFeedStorage': 1,
 'finish_reason': 'finished',
 'finish_time': datetime.datetime(2026, 11, 24, 13, 10, 7, 998007, tzinfo=datetime.timezone.utc),
 'httpcompression/response_bytes': 1586000,
 'httpcompression/response_count': 2,
 'item_scraped_count': 1,
 'log_count/INFO': 11,
 'memusage/max': 69513216,
 'memusage/startup': 69513216,
 'response_received_count': 2,
 'robotstxt/request_count': 1,
 'robotstxt/response_count': 1,
 'robotstxt/response_

[
{"title": "franceinfo - Actualités en temps réel et info en direct", "all_links": {"Sports": "https://www.franceinfo.fr/sports/", "Culture": "https://www.franceinfo.fr/culture/", "Vrai ou faux": "https://www.franceinfo.fr/vrai-ou-fake/", "Politique": "https://www.franceinfo.fr/politique/", "Société": "https://www.franceinfo.fr/societe/", "Environnement": "https://www.franceinfo.fr/environnement/", "Monde": "https://www.franceinfo.fr/monde/"}}
]

On veut ensuite *entrer* dans les liens des différentes sous-catégories
pour récupérer les articles. Pour cela, nous créons une méthode
`parse_category()` prend en argument un objet `Response` qui sera la
réponse correspondant aux liens des régions. On peut comme ceci
traverser un site en définissant des méthodes différentes en fonction du
type de contenu.

Si la structure du site est plus profonde, on peut empiler autant de
couches que souhaité.

Quand on arrive sur une page d'une sous-catégorie, on peut vouloir
récupérer tous les éléments de la page. Pour cela, on réutilise le
scrapy Shell pour commencer le développement de la nouvelle méthode
d'extraction.

Par exemple pour la page `https://www.lemonde.fr/international/` :

```
scrapy shell 'https://www.lemonde.fr/international/'
```

Le fil des articles est stocké dans une balise avec la classe
`class=river`.

```
In [3]: response.css(".river")
Out[3]:
[<Selector xpath="descendant-or-self::*[@class and contains(concat(' ', normalize-space(@class), ' '), ' fleuve ')]" data='<div class="fleuve">\n   <section>\n      '>,
<Selector xpath="descendant-or-self::*[@class and contains(concat(' ', normalize-space(@class), ' '), ' fleuve ')]" data='<div class="fleuve">\n</div>'>]
```

Pour récupérer chacun des articles, il faut adresser les balises
`<article>` contenues dans le sélecteur:

```
In [4]: response.css(".river")[0].css(".teaser")
Out[4]:
[<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi mg'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>]   
```

Comme précédemment, on peut empiler les sélecteurs `css` pour créer des
requêtes plus complexes.

Par exemple, pour récupérer tous les titres des différents articles :

```
In [8]: response.css(".river")[0].css(".teaser h3::text").extract()
Out[8]:
['«\xa0Le Livre noir de Gaza\xa0»\xa0: décryptage d’une entreprise d’anéantissement',
 'Au Maroc, la rue se mobilise pour Gaza, le pouvoir maintient sa coopération avec Israël',
 'La guerre à Gaza ravive de manière inédite le traumatisme des Palestiniens',
 '7-Octobre\xa0: Benyamin Nétanyahou sous la menace de la justice internationale',
 'En France, un mouvement propalestinien phagocyté par LFI',
 'En direct, guerre au\xa0Proche-Orient\xa0: le Hezbollah dit soutenir les efforts du Liban en vue d’un cessez-le-feu',
 'A Gaza, des destructions agricoles sans précédent qui accroissent le risque de famine',
 'La Chine veut croire à la reprise de son économie',
 'Comment «\xa0Le Monde\xa0» a couvert un an de guerre à Gaza',
 'Guerre Israël-Hamas\xa0: «\xa0L’enjeu consiste à admettre que le salut ne vient jamais de la puissance, uniquement de la négociation\xa0»',
 'En cartes\xa0: un an de destructions dans la bande de Gaza', 
...]
```

En HTML les données sont souvent de très mauvaise qualité. Il faut
définir des méthodes permettant de les nettoyer pour être intégrées dans
des bases de données.

Par exemple, pour supprimer tous les espaces superflus :

In [17]:
def clean_spaces(string_):
    if string_ is not None:
        return " ".join(string_.split())

Pour l'appliquer à tous les titres récupérés, on peut faire une list
comprehension : 
```
In [11]: [clean_spaces(article) for article in response.css(".river")[0].css(".teaser h3::text").extract()]  

Out[11]: ['« Le Livre noir de Gaza » : décryptage d’une entreprise d’anéantissement',
 'Au Maroc, la rue se mobilise pour Gaza, le pouvoir maintient sa coopération avec Israël',
 'La guerre à Gaza ravive de manière inédite le traumatisme des Palestiniens',
 '7-Octobre : Benyamin Nétanyahou sous la menace de la justice internationale',
 'En France, un mouvement propalestinien phagocyté par LFI',
 'En direct, guerre au Proche-Orient : le Hezbollah dit soutenir les efforts du Liban en vue d’un cessez-le-feu',
 'A Gaza, des destructions agricoles sans précédent qui accroissent le risque de famine',
 'La Chine veut croire à la reprise de son économie',
...
]

```

La méthode précédente est intéressante si l'on ne recherche qu'une seule
information par article.

Par contre si l'on veut récupérer d'autres caractéristiques comme
l'image ou la description par exemple, il est plus intéressant et plus
efficace de récupérer l'objet et d'effectuer plusieurs traitements sur
ce dernier.

Chaque objet retourné par les requêtes `css` est un selecteur avec
lequel on peut interagir.

Par exemple pour récupérer le titre, l'image et la description :

```
In [19]: for article in response.css(".river")[0].css(".teaser"):
    ...:     title = clean_spaces(article.css('h3::text').extract_first())
    ...:     image = article.css('img::attr(data-src)').extract_first()
    ...:     description = article.css('p::text').extract_first()
    ...:     print(f"Title: {title} \nImage: {image} \nDescription: {description}\n-----")
```

```
Title: Le Nobel de médecine honore deux pionniers de la régulation des gènes par de tout petits ARN 
Image: https://img.lemde.fr/2024/10/07/35/0/3000/2000/110/73/60/0/6dafcf3_1728304381125-063-2176426487.jpg 
Description: Les deux lauréats américains, Victor Ambros et Gary Ruvkun, sont récompensés pour l’identification d’une voie de contrôle de l’activité des gènes par des micro-ARN, qui est présente dans toutes les cellules animales et végétales.
-----
Title: La position de La France insoumise sur le 7-Octobre, cause de fractures à gauche 
Image: https://img.lemde.fr/2024/10/05/0/0/5304/3536/110/73/60/0/9c38cc7_1728124287163-000-34l44a7.jpg 
Description: « 7-Octobre, un an après ». Alors que Jean-Luc Mélenchon et ses troupes ont érigé la cause palestinienne comme un axe de campagne électorale, les socialistes ont condamné le refus des « insoumis » de qualifier le Hamas de mouvement « terroriste ».
-----
```


> **Adaptation franceinfo** — Sur une page de rubrique (ex : `https://www.franceinfo.fr/monde/`), chaque article est une carte `<article class="card-article-base">` : titre dans `.card-article-base__title`, description dans `.card-article-base__chapo`, image dans `img` (attribut `data-src` car l'image est chargée en différé) :

In [18]:
r = requests.get("https://www.franceinfo.fr/monde/", headers=UA, timeout=10)
category_response = HtmlResponse(url=r.url, body=r.content, encoding="utf-8")

for article in category_response.css("article.card-article-base")[:3]:
    title = clean_spaces(" ".join(article.css(".card-article-base__title::text").extract()))
    image = article.css("img::attr(data-src)").extract_first() or article.css("img::attr(src)").extract_first()
    description = clean_spaces(article.css(".card-article-base__chapo::text").extract_first())
    print(f"Title: {title} \nImage: {image} \nDescription: {description}\n-----")

Title: A Gaza, le responsable de la sécurité intérieure du Hamas tué par une frappe israélienne 
Image: https://www.franceinfo.fr/pictures/8f8tPbedR48BjaEnEjoEJyVMrjU/0x0:992x682/432x297/filters:format(jpg)/2026/11/24/6b058ad7b3b97752754914.jpg 
Description: L'armée et les services du renseignement intérieur israéliens avaient annoncé peu avant avoir "frappé un terroriste de haut rang" du mouvement, sans préciser son identité.
-----
Title: Un ferry flambant neuf heurte violemment les rochers du lac Victoria 
Image: https://www.franceinfo.fr/pictures/r7aeWr-Rv-aZt206EhhWWXUQOgw/0x0:1080x1920/fit-in/432x243/filters:upscale():fill(blur):format(jpg)/2026/11/24/6b057c2001b1b469489357.png 
Description: None
-----
Title: "J'ai découvert que le démon rôdait" : le journal du pape François publié par un ancien proche collaborateur révèle sa foi profonde, mais aussi ses travers et ses fragilités 
Image: https://www.franceinfo.fr/pictures/OOWf2oPUBD69nGYITpXClZkroQ8/0x384:4500x2915/432x243/filters

## Persistence des données

Pour pouvoir stocker les informations que l'on récupère en parcourant un
site, il faut pouvoir les stocker. On utilise soit de simples
dictionnaires Python, ou mieux des `scrapy.Item` qui sont des
dictionnaires améliorés.

Nous allons voir les deux façons de faire. On peut réécrire la méthode
`parse_category()` pour lui faire retourner un dictionnaire
correspondant à chaque article rencontré :


In [19]:
def parse_category(self, response):
    for article in response.css(".river")[0].css(".teaser"):
        title = self.clean_spaces(article.css("h3::text").extract_first())
        image = article.css("img::attr(data-src)").extract_first()
        description = article.css("p::text").extract_first()
        yield {
            "title": title,
            "image": image,
            "description": description
        }

Si on combine tout dans la spider :

In [20]:
# %load newscrawler/newscrawler/spiders/lemonde_v3.py
import scrapy
from scrapy import Request


class LemondeSpider(scrapy.Spider):
    name = "lemondev3"
    allowed_domains = ["www.lemonde.fr"]
    start_urls = ['https://www.lemonde.fr']

    def parse(self, response):
        title = response.css('title::text').extract_first()
        all_links = {
            name: response.urljoin(url) for name, url in zip(
                response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
                response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
        }
        for link in all_links.values():
            yield Request(link, callback=self.parse_category)

    def parse_category(self, response):
        for article in response.css(".river")[0].css(".teaser"):
            title = self.clean_spaces(article.css("h3::text").extract_first())
            image = article.css("img::attr(data-src)").extract_first()
            description = article.css(".txt3::text").extract_first()
            yield {
                "title": title,
                "image": image,
                "description": description
            }

    def clean_spaces(self, string):
        if string:
            return " ".join(string.split())

Version franceinfo (`franceinfo_v3.py`) : on transmet aussi le nom de la rubrique (`cb_kwargs`) et l'URL de l'article.

In [21]:
!cat newscrawler/newscrawler/spiders/franceinfo_v3.py

import scrapy
from scrapy import Request


class FranceinfoSpider(scrapy.Spider):
    """Équivalent de lemonde_v3.py : on entre dans chaque rubrique et on
    récupère les articles listés (titre, image, description)."""
    name = "franceinfov3"
    allowed_domains = ["www.franceinfo.fr"]
    start_urls = ["https://www.franceinfo.fr/"]
    # Lien du bloc "rubriques" qui n'est pas une rubrique d'actualité
    excluded = ["/une-information-transparente-franceinfo/"]

    def parse(self, response):
        for link in response.css("#rubriques a.title-link__link"):
            if link.attrib["href"] in self.excluded:
                continue
            category = self.clean_spaces(" ".join(link.css("*::text").extract()))
            # cb_kwargs transmet le nom de la rubrique à la méthode de callback
            yield Request(
                response.urljoin(link.attrib["href"]),
                callback=self.parse_category,
                cb_kwargs={"category": category},
            )


On peut maintenant lancer notre spider avec la commande suivante :
```
scrapy crawl <SPIDER_NAME>
```

`scrapy crawl` permet de démarrer le processus en allant chercher la
classe `scrapy.Spider` dont l'attribut `name` = &lt;NAME&gt;.

Par exemple, pour la spider `LeMondeSpider` :

In [22]:
!cd newscrawler && scrapy crawl franceinfov3 -L INFO

2026-11-24 13:10:08 [scrapy.utils.log] INFO: Scrapy 2.11.2 started (bot: newscrawler)
2026-11-24 13:10:08 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.12.9, cssselect 1.3.0, parsel 1.10.0, w3lib 2.3.1, Twisted 25.5.0, Python 3.12.16 (main, Nov  5 2026, 08:06:14) [GCC 14.2.0], pyOpenSSL 25.3.0 (OpenSSL 3.5.4 30 Sep 2025), cryptography 46.0.3, Platform Linux-7.0.14-linuxkit-aarch64-with-glibc2.41
2026-11-24 13:10:08 [scrapy.addons] INFO: Enabled addons:
[]
2026-11-24 13:10:08 [scrapy.extensions.telnet] INFO: Telnet Password: 4a9fd54fbcfe744d
2026-11-24 13:10:08 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.logstats.LogStats']
2026-11-24 13:10:08 [scrapy.crawler] INFO: Overridden settings:
{'BOT_NAME': 'newscrawler',
 'DOWNLOAD_DELAY': 1,
 'FEED_EXPORT_ENCODING': 'utf-8',
 'LOG_LEVEL': 'INFO',
 'NEWSPIDER_MODULE': 'newscrawler.sp

2026-11-24 13:10:08 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.offsite.OffsiteMiddleware',
 'scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
 'scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
 'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
 'scrapy.downloadermiddlewares.defaultheaders.DefaultHeadersMiddleware',
 'scrapy.downloadermiddlewares.useragent.UserAgentMiddleware',
 'scrapy.downloadermiddlewares.retry.RetryMiddleware',
 'scrapy.downloadermiddlewares.redirect.MetaRefreshMiddleware',
 'scrapy.downloadermiddlewares.httpcompression.HttpCompressionMiddleware',
 'scrapy.downloadermiddlewares.redirect.RedirectMiddleware',
 'scrapy.downloadermiddlewares.cookies.CookiesMiddleware',
 'scrapy.downloadermiddlewares.httpproxy.HttpProxyMiddleware',
 'scrapy.downloadermiddlewares.stats.DownloaderStats']
2026-11-24 13:10:08 [scrapy.middleware] INFO: Enabled spider middlewares:
['scrapy.spidermiddlew

2026-11-24 13:10:18 [scrapy.core.engine] INFO: Closing spider (finished)
2026-11-24 13:10:18 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 2297,
 'downloader/request_count': 9,
 'downloader/request_method_count/GET': 9,
 'downloader/response_bytes': 918998,
 'downloader/response_count': 9,
 'downloader/response_status_count/200': 9,
 'elapsed_time_seconds': 9.455549,
 'finish_reason': 'finished',
 'finish_time': datetime.datetime(2026, 11, 24, 13, 10, 18, 82671, tzinfo=datetime.timezone.utc),
 'httpcompression/response_bytes': 6632486,
 'httpcompression/response_count': 9,
 'item_scraped_count': 149,
 'log_count/INFO': 10,
 'memusage/max': 69054464,
 'memusage/startup': 69054464,
 'request_depth_max': 1,
 'response_received_count': 9,
 'robotstxt/request_count': 1,
 'robotstxt/response_count': 1,
 'robotstxt/response_status_count/200': 1,
 'scheduler/dequeued': 8,
 'scheduler/dequeued/memory': 8,
 'scheduler/enqueued': 8,
 'scheduler/enqueued/memory'

On peut exporter les résultats de ces retours dans différents formats de
fichiers.

-   CSV : `scrapy crawl lemondev3 -o lemonde.csv`
-   JSON : `scrapy crawl lemondev3 -o lemonde.json`
-   JSONLINE : `scrapy crawl lemondev3 -o lemonde.jl`
-   XML : `scrapy crawl lemondev3 -o lemonde.xml`

### Exercice :

Exécuter la spider avec les différents formats de stockage.
Explorer ensuite le contenu des fichiers ainsi créés.

> **Réponse** — On lance la spider une fois par format. On utilise `-O` (majuscule) qui **remplace** le fichier, alors que `-o` **ajoute** à la fin d'un fichier existant (ce qui casse un fichier JSON si on relance).

In [23]:
for ext in ["csv", "json", "jl", "xml"]:
    !cd newscrawler && scrapy crawl franceinfov3 -L WARNING -O ../data/franceinfo.{ext}

!ls -la data/

total 544
drwxr-xr-x 8 root root    256 Nov 24 13:10 .
drwxr-xr-x 9 root root    288 Nov 24 13:07 ..
-rw-r--r-- 1 root root  10409 Nov 24 13:10 citation.json
-rw-r--r-- 1 root root  88477 Nov 24 13:10 franceinfo.csv
-rw-r--r-- 1 root root  98144 Nov 24 13:10 franceinfo.jl
-rw-r--r-- 1 root root  98295 Nov 24 13:10 franceinfo.json
-rw-r--r-- 1 root root 102575 Nov 24 13:10 franceinfo.xml
-rw-r--r-- 1 root root    454 Nov 24 13:10 franceinfo_v2.json


In [24]:
print("--- CSV : une ligne par article, colonnes séparées par des virgules")
!head -n 3 data/franceinfo.csv
print("\n--- JSON : une liste [ ... ] de dictionnaires")
!head -c 600 data/franceinfo.json
print("\n\n--- JSON Lines : un dictionnaire JSON par ligne (pratique pour les gros volumes)")
!head -n 2 data/franceinfo.jl
print("\n--- XML : chaque article dans une balise <item>")
!head -c 700 data/franceinfo.xml

--- CSV : une ligne par article, colonnes séparées par des virgules






--- JSON : une liste [ ... ] de dictionnaires
[
{"category": "Sports", "title": "Anorexie, dépression... La belle revanche de Marjolaine Pierré, sacrée championne du monde de triathlon longue distance", "image": "https://www.franceinfo.fr/pictures/csp9qE67pz0Uyac0ynNAX_KNgf4/838x0:2209x2438/fit-in/432x243/filters:upscale():fill(blur):format(jpg)/2026/11/24/6b056392ba5d2597765907.jpg", "description": "La triathlète française, déjà titrée en 2023 et 2025, a décroché une troisième couronne mondiale, dimanche, en triathlon longue distance à Abou Dhabi.", "url": "https://www.franceinfo.fr/sports/triathlon/anorexie-depression-la-bel



--- JSON Lines : un dictionnaire JSON par ligne (pratique pour les gros volumes)
{"category": "Sports", "title": "Anorexie, dépression... La belle revanche de Marjolaine Pierré, sacrée championne du monde de triathlon longue distance", "image": "https://www.franceinfo.fr/pictures/csp9qE67pz0Uyac0ynNAX_KNgf4/838x0:2209x2438/fit-in/432x243/filters:upscale():fill(blur):format(jpg)/2026/11/24/6b056392ba5d2597765907.jpg", "description": "La triathlète française, déjà titrée en 2023 et 2025, a décroché une troisième couronne mondiale, dimanche, en triathlon longue distance à Abou Dhabi.", "url": "https://www.franceinfo.fr/sports/triathlon/anorexie-depression-la-belle-revanche-de-marjolaine-pierre-sacree-championne-du-monde-de-triathlon-longue-distance_8297409.html"}
{"category": "Sports", "title": "Stellantis quitte la compétition sportive", "image": "https://www.franceinfo.fr/pictures/Kzj9hLyRDSb_QLvGGRPeNvVpfzk/0x32:1024x608/432x243/filters:format(jpg)/2026/11/24/6b0540949bfef714168125.j


--- XML : chaque article dans une balise <item>
<?xml version="1.0" encoding="utf-8"?>
<items>
<item><category>Sports</category><title>Anorexie, dépression... La belle revanche de Marjolaine Pierré, sacrée championne du monde de triathlon longue distance</title><image>https://www.franceinfo.fr/pictures/csp9qE67pz0Uyac0ynNAX_KNgf4/838x0:2209x2438/fit-in/432x243/filters:upscale():fill(blur):format(jpg)/2026/11/24/6b056392ba5d2597765907.jpg</image><description>La triathlète française, déjà titrée en 2023 et 2025, a décroché une troisième couronne mondiale, dimanche, en triathlon longue distance à Abou Dhabi.</description><url>https://www.franceinfo.fr/sports/triathlon/anorexie-depression-la-belle-revanche-de-marjolaine-pierre-sac

In [25]:
# Les fichiers se relisent facilement avec pandas
articles = pd.read_json("data/franceinfo.jl", lines=True)
print(len(articles), "articles")
print(articles["category"].value_counts())
articles.head()

149 articles
category
Vrai ou faux    33
Société         28
Politique       25
Monde           24
Culture         20
Sports          19
Name: count, dtype: int64


,category,title,image,description,url
0,Sports,"Anorexie, dépression... La belle revanche de M...",https://www.franceinfo.fr/pictures/csp9qE67pz0...,"La triathlète française, déjà titrée en 2023 e...",https://www.franceinfo.fr/sports/triathlon/ano...
1,Sports,Stellantis quitte la compétition sportive,https://www.franceinfo.fr/pictures/Kzj9hLyRDSb...,L’annonce marque les esprits dans le sport aut...,https://www.franceinfo.fr/replay-radio/le-brie...
2,Sports,L'UE veut priver les athlètes russes de compét...,https://www.franceinfo.fr/pictures/c12kcNqhEqj...,L'UE envisage d'interdire les athlètes russes ...,https://www.franceinfo.fr/replay-radio/d-un-mo...
3,Sports,Route du Rhum : Armel Le Cléac'h raconte comme...,https://www.franceinfo.fr/pictures/vNha07NKkvq...,"Le 4 novembre, alors qu'il était deuxième de l...",https://www.franceinfo.fr/sports/voile/route-d...
4,Sports,"José Mourinho : ""Si un joueur fait grève car o...",https://www.franceinfo.fr/pictures/WcqnOiBenGk...,L'entraîneur du Real Madrid s'est exprimé sur ...,https://www.franceinfo.fr/sports/foot/jose-mou...


> Les 4 fichiers contiennent les mêmes données. **CSV** s'ouvre dans Excel mais aplatit tout en texte ; **JSON** garde la structure mais doit être lu en entier ; **JSON Lines** se lit ligne par ligne (idéal pour de gros volumes ou pour ajouter des données) ; **XML** est plus verbeux, utile pour des outils qui l'exigent.

## Votre premier Item

La classe `Item` permet de structurer les données que l'on souhaite
récupérer sous la forme d'un modèle. Les items doivent être définis dans
le fichier `items.py` créé par la commande `scrapy startproject`. Les
`Item` héritent de la class `scrapy.Item`.

On veut structurer les données avec trois champs : le titre, l'image et la description. Scrapy utilise une classe `scrapy.Field` permettant de 'déclarer' ces champs. Dans notre cas :

In [26]:
# %load newscrawler/newscrawler/items.py

# Define here the models for your scraped items
#
# See documentation in:
# https://docs.scrapy.org/en/latest/topics/items.html

import scrapy


class ArticleItem(scrapy.Item):
    title = scrapy.Field()
    image = scrapy.Field()
    description = scrapy.Field()

Utiliser la classe `scrapy.Item` plutôt qu'un simple dictionnaire permet
plus de contrôle sur la structure des données. En effet, on ne peut
insérer dans les items que des données avec des clés 'déclarées'. Ce qui
assure une plus grande cohérence au sein d'un projet.

On peut instancier un item de plusieurs façons :

In [27]:
article_item = ArticleItem(title="Yann Lecun and Meta sortent Llama 3.2", image=None, description="LLama 3.2 n'est pas disponible en france")

In [28]:
print(article_item)

{'description': "LLama 3.2 n'est pas disponible en france",
 'image': None,
 'title': 'Yann Lecun and Meta sortent Llama 3.2'}


In [29]:
article_item = ArticleItem()
article_item["title"] = 'Léon Marchand recoit la légion d\'honneur'
article_item["description"] = 'Quadruple champion olympic'

In [30]:
print(article_item)

{'description': 'Quadruple champion olympic',
 'title': "Léon Marchand recoit la légion d'honneur"}


La définition d'un item permet de palier toutes les erreurs de typo dans
les champs.

In [31]:
article_item = ArticleItem()
try:
    article_item["titelkjwnxvmnscbvmknxc"] = 'Erreur exemple'
except KeyError as error:
    print("KeyError :", error)  # le champ n'est pas déclaré dans ArticleItem

KeyError : 'ArticleItem does not support field: titelkjwnxvmnscbvmknxc'


Les items héritent des dictionnaires Python, et possèdent donc toutes
les méthodes de ceux-ci :

In [32]:
article_item = ArticleItem(title="Nouvelle lamborghini")
print(article_item["title"])  # Méthode __getitem__()
print(article_item.get("description", "no description provided"))  # Méthode get()

Nouvelle lamborghini
no description provided


On peut transformer un `Item` en dictionnaire très facilement, en le
passant au constructeur:

In [33]:
article_item = ArticleItem(title="Drone DJI")
print(type(article_item))
dict_item = dict(article_item)
print(type(dict_item))
print(dict_item)

<class '__main__.ArticleItem'>
<class 'dict'>
{'title': 'Drone DJI'}


On intègre maintenant cet item dans notre spider.

In [34]:
# %load newscrawler/newscrawler/spiders/lemonde_v4.py

class LemondeSpider(scrapy.Spider):
    name = "lemondev4"
    allowed_domains = ["www.lemonde.fr"]
    start_urls = ['https://www.lemonde.fr']

    def parse(self, response, **kwargs):
        all_links = {
            name: response.urljoin(url) for name, url in zip(
                response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
                response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
        }
        for link in all_links.values():
            yield Request(link, callback=self.parse_category)

    def parse_category(self, response):
        for article in response.css(".river")[0].css(".teaser"):
            title = self.clean_spaces(article.css("h3::text").extract_first())
            image = article.css("img::attr(data-src)").extract_first()
            description = article.css("p::text").extract_first()
            yield ArticleItem(
                title=title,
                image=image,
                description=description
            )

    def clean_spaces(self, string):
        if string:
            return " ".join(string.split())

Version franceinfo (`franceinfo_v4.py`) et l'item utilisé (`items.py`, avec deux champs ajoutés : `category` et `url`) :

In [35]:
!cat newscrawler/newscrawler/spiders/franceinfo_v4.py
!cat newscrawler/newscrawler/items.py

import scrapy
from scrapy import Request

from ..items import ArticleItem


class FranceinfoSpider(scrapy.Spider):
    """Équivalent de lemonde_v4.py : les articles sont renvoyés sous forme
    d'ArticleItem (voir items.py) et passent par les pipelines (voir settings.py)."""
    name = "franceinfov4"
    allowed_domains = ["www.franceinfo.fr"]
    start_urls = ["https://www.franceinfo.fr/"]
    # Lien du bloc "rubriques" qui n'est pas une rubrique d'actualité
    excluded = ["/une-information-transparente-franceinfo/"]

    def parse(self, response, **kwargs):
        for link in response.css("#rubriques a.title-link__link"):
            if link.attrib["href"] in self.excluded:
                continue
            category = " ".join(" ".join(link.css("*::text").extract()).split())
            yield Request(
                response.urljoin(link.attrib["href"]),
                callback=self.parse_category,
                cb_kwargs={"category": category},
            )

    def parse_

# Define here the models for your scraped items
#
# See documentation in:
# https://docs.scrapy.org/en/latest/topics/items.html

import scrapy


class ArticleItem(scrapy.Item):
    title = scrapy.Field()
    image = scrapy.Field()
    description = scrapy.Field()
    # Champs ajoutés pour la version franceinfo
    category = scrapy.Field()
    url = scrapy.Field()


On voit bien que le générateur retourne maintenant un `Item`.

### Exercice : 

Relancer la spider pour vérifier le bon déroulement de l'extraction.


> **Réponse** — On relance `franceinfov4` : les statistiques de fin (`item_scraped_count`, `finish_reason: finished`) montrent le bon déroulement, et la sortie est maintenant un `ArticleItem` (affiché `{'category': ..., 'title': ...}` dans les logs en mode DEBUG).

In [36]:
!cd newscrawler && scrapy crawl franceinfov4 -L INFO

2026-11-24 13:10:59 [scrapy.utils.log] INFO: Scrapy 2.11.2 started (bot: newscrawler)
2026-11-24 13:10:59 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.12.9, cssselect 1.3.0, parsel 1.10.0, w3lib 2.3.1, Twisted 25.5.0, Python 3.12.16 (main, Nov  5 2026, 08:06:14) [GCC 14.2.0], pyOpenSSL 25.3.0 (OpenSSL 3.5.4 30 Sep 2025), cryptography 46.0.3, Platform Linux-7.0.14-linuxkit-aarch64-with-glibc2.41
2026-11-24 13:10:59 [scrapy.addons] INFO: Enabled addons:
[]
2026-11-24 13:10:59 [scrapy.extensions.telnet] INFO: Telnet Password: 4e4d80d955f8df76
2026-11-24 13:10:59 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.logstats.LogStats']
2026-11-24 13:10:59 [scrapy.crawler] INFO: Overridden settings:
{'BOT_NAME': 'newscrawler',
 'DOWNLOAD_DELAY': 1,
 'FEED_EXPORT_ENCODING': 'utf-8',
 'LOG_LEVEL': 'INFO',
 'NEWSPIDER_MODULE': 'newscrawler.sp

2026-11-24 13:10:59 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.offsite.OffsiteMiddleware',
 'scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
 'scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
 'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
 'scrapy.downloadermiddlewares.defaultheaders.DefaultHeadersMiddleware',
 'scrapy.downloadermiddlewares.useragent.UserAgentMiddleware',
 'scrapy.downloadermiddlewares.retry.RetryMiddleware',
 'scrapy.downloadermiddlewares.redirect.MetaRefreshMiddleware',
 'scrapy.downloadermiddlewares.httpcompression.HttpCompressionMiddleware',
 'scrapy.downloadermiddlewares.redirect.RedirectMiddleware',
 'scrapy.downloadermiddlewares.cookies.CookiesMiddleware',
 'scrapy.downloadermiddlewares.httpproxy.HttpProxyMiddleware',
 'scrapy.downloadermiddlewares.stats.DownloaderStats']
2026-11-24 13:10:59 [scrapy.middleware] INFO: Enabled spider middlewares:
['scrapy.spidermiddlew

2026-11-24 13:11:08 [scrapy.core.engine] INFO: Closing spider (finished)
2026-11-24 13:11:08 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 2297,
 'downloader/request_count': 9,
 'downloader/request_method_count/GET': 9,
 'downloader/response_bytes': 918977,
 'downloader/response_count': 9,
 'downloader/response_status_count/200': 9,
 'elapsed_time_seconds': 9.304361,
 'finish_reason': 'finished',
 'finish_time': datetime.datetime(2026, 11, 24, 13, 11, 8, 963211, tzinfo=datetime.timezone.utc),
 'httpcompression/response_bytes': 6632483,
 'httpcompression/response_count': 9,
 'item_scraped_count': 149,
 'log_count/INFO': 10,
 'memusage/max': 69042176,
 'memusage/startup': 69042176,
 'request_depth_max': 1,
 'response_received_count': 9,
 'robotstxt/request_count': 1,
 'robotstxt/response_count': 1,
 'robotstxt/response_status_count/200': 1,
 'scheduler/dequeued': 8,
 'scheduler/dequeued/memory': 8,
 'scheduler/enqueued': 8,
 'scheduler/enqueued/memory'

## Postprocessing

Si l'on se réfère au diagramme d'architecture de Scrapy, on voit qu'il
est possible d'insérer des composants supplémentaires dans le flux de
traitement. Ces composants s'appellent `Pipelines`.

Par défaut, tous les `Item` générés au sein d'un projet Scrapy passent
par les `Pipelines`. Les pipelines sont utilisés la plupart du temps
pour :

-   Nettoyer du contenu HTML ;
-   Valider les données scrapées ;
-   Supprimer les items qu'on ne souhaite pas stocker ;
-   Stocker ces objets dans des bases de données.

Les pipelines doivent être définis dans le fichier `pipelines.py`.

Dans notre cas, on peut vouloir nettoyer le champ `title` pour enlever
les caractères superflus.

Nous allons alors transferer la fonction de nettoyage du code html dans
une Pipeline.

In [37]:
# %load newscrawler/newscrawler/pipelines.py

# Define your item pipelines here
#
# Don't forget to add your pipeline to the ITEM_PIPELINES setting
# See: https://docs.scrapy.org/en/latest/topics/item-pipeline.html

from scrapy.exceptions import DropItem


class TextPipeline(object):

    def process_item(self, item, spider):
        if item['title']:
            item["title"] = clean_spaces(item["title"])
            return item
        else:
            raise DropItem(f"Missing title in {item}")


def clean_spaces(string):
    if string:
        return " ".join(string.split())

Le fichier `pipelines.py` du projet (le `TextPipeline` nettoie aussi la description, et le `MongoPipeline` y est prêt pour la partie 3Mongo) et les réglages ajoutés à `settings.py` :

In [38]:
!cat newscrawler/newscrawler/pipelines.py
!tail -n 13 newscrawler/newscrawler/settings.py

# Define your item pipelines here
#
# Don't forget to add your pipeline to the ITEM_PIPELINES setting
# See: https://docs.scrapy.org/en/latest/topics/item-pipeline.html

from scrapy.exceptions import DropItem


class TextPipeline(object):

    def process_item(self, item, spider):
        if item.get("title") and clean_spaces(item["title"]):
            item["title"] = clean_spaces(item["title"])
            if item.get("description"):
                item["description"] = clean_spaces(item["description"])
            return item
        else:
            raise DropItem(f"Missing title in {item}")


class MongoPipeline(object):
    """Stocke les items dans MongoDB (vu dans la partie 3Mongo).
    Non activé dans settings.py tant qu'aucun serveur Mongo ne tourne."""

    collection_name = "scrapy_items"

    def open_spider(self, spider):
        import pymongo
        self.client = pymongo.MongoClient()
        self.db = self.client["franceinfo"]

    def close_spider(self, spider):
   

#HTTPCACHE_STORAGE = 'scrapy.extensions.httpcache.FilesystemCacheStorage'

# --- Ajouts du cours (2Scrapy) ---
# Les items passent par le TextPipeline (nettoyage du titre et de la description)
ITEM_PIPELINES = {
    "newscrawler.pipelines.TextPipeline": 300,
}
# On reste poli avec le site : 1 seconde entre deux requêtes
DOWNLOAD_DELAY = 1
# Accents lisibles dans les exports JSON
FEED_EXPORT_ENCODING = "utf-8"
# Version recommandée du calcul d'empreinte des requêtes (évite un avertissement)
REQUEST_FINGERPRINTER_IMPLEMENTATION = "2.7"


Pour dire au process Scrapy de faire transiter les items par ces
pipelines. Il faut le spécifier dans le fichier de paramétrage
`settings.py`.

In [39]:
ITEM_PIPELINES = {
    'newscrawler.pipelines.TextPipeline': 300,
}

On peut maintenant supprimer la fonction `clean_spaces()` de
l'extraction des données et laisser le Pipeline faire son travail. La
valeur entière définie permet de déterminer l'ordre dans lequel les
pipelines vont être appelés. Ces entiers peuvent être compris entre 0 et
1000.

On relance notre spider :
```
scrapy crawl lemondev4 -o ../data/articles.json
```

In [40]:
!cd newscrawler && scrapy crawl franceinfov4 -L INFO -O ../data/articles.json

articles = pd.read_json("data/articles.json")
print(len(articles), "articles après le pipeline")
articles[["category", "title", "description"]].head(10)

2026-11-24 13:11:09 [scrapy.utils.log] INFO: Scrapy 2.11.2 started (bot: newscrawler)
2026-11-24 13:11:09 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.12.9, cssselect 1.3.0, parsel 1.10.0, w3lib 2.3.1, Twisted 25.5.0, Python 3.12.16 (main, Nov  5 2026, 08:06:14) [GCC 14.2.0], pyOpenSSL 25.3.0 (OpenSSL 3.5.4 30 Sep 2025), cryptography 46.0.3, Platform Linux-7.0.14-linuxkit-aarch64-with-glibc2.41
2026-11-24 13:11:09 [scrapy.addons] INFO: Enabled addons:
[]
2026-11-24 13:11:09 [scrapy.extensions.telnet] INFO: Telnet Password: fc9cf5dcf1768704
2026-11-24 13:11:09 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-11-24 13:11:09 [scrapy.crawler] INFO: Overridden settings:
{'BOT_NAME': 'newscrawler',
 'DOWNLOAD_DELAY': 1,
 'FEED_EXPORT_ENCODING': 'utf-8',
 'LOG_LEVEL'

2026-11-24 13:11:09 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.offsite.OffsiteMiddleware',
 'scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
 'scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
 'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
 'scrapy.downloadermiddlewares.defaultheaders.DefaultHeadersMiddleware',
 'scrapy.downloadermiddlewares.useragent.UserAgentMiddleware',
 'scrapy.downloadermiddlewares.retry.RetryMiddleware',
 'scrapy.downloadermiddlewares.redirect.MetaRefreshMiddleware',
 'scrapy.downloadermiddlewares.httpcompression.HttpCompressionMiddleware',
 'scrapy.downloadermiddlewares.redirect.RedirectMiddleware',
 'scrapy.downloadermiddlewares.cookies.CookiesMiddleware',
 'scrapy.downloadermiddlewares.httpproxy.HttpProxyMiddleware',
 'scrapy.downloadermiddlewares.stats.DownloaderStats']
2026-11-24 13:11:09 [scrapy.middleware] INFO: Enabled spider middlewares:
['scrapy.spidermiddlew

2026-11-24 13:11:20 [scrapy.core.engine] INFO: Closing spider (finished)
2026-11-24 13:11:20 [scrapy.extensions.feedexport] INFO: Stored json feed (149 items) in: ../data/articles.json
2026-11-24 13:11:20 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 2297,
 'downloader/request_count': 9,
 'downloader/request_method_count/GET': 9,
 'downloader/response_bytes': 918973,
 'downloader/response_count': 9,
 'downloader/response_status_count/200': 9,
 'elapsed_time_seconds': 10.959149,
 'feedexport/success_count/FileFeedStorage': 1,
 'finish_reason': 'finished',
 'finish_time': datetime.datetime(2026, 11, 24, 13, 11, 20, 530552, tzinfo=datetime.timezone.utc),
 'httpcompression/response_bytes': 6632478,
 'httpcompression/response_count': 9,
 'item_scraped_count': 149,
 'log_count/INFO': 11,
 'memusage/max': 69386240,
 'memusage/startup': 69386240,
 'request_depth_max': 1,
 'response_received_count': 9,
 'robotstxt/request_count': 1,
 'robotstxt/response_count

149 articles après le pipeline


,category,title,description
0,Sports,"Anorexie, dépression... La belle revanche de M...","La triathlète française, déjà titrée en 2023 e..."
1,Sports,Stellantis quitte la compétition sportive,L’annonce marque les esprits dans le sport aut...
2,Sports,L'UE veut priver les athlètes russes de compét...,L'UE envisage d'interdire les athlètes russes ...
3,Sports,Route du Rhum : Armel Le Cléac'h raconte comme...,"Le 4 novembre, alors qu'il était deuxième de l..."
4,Sports,"José Mourinho : ""Si un joueur fait grève car o...",L'entraîneur du Real Madrid s'est exprimé sur ...
5,Sports,"""Ça ressemble vraiment à une petite montagne"" ...","Plus puissante que Jaws à Hawaï, plus haute qu..."
6,Sports,"""C'est comme le patinage artistique, mais en l...","Domitille Kiger, co-organise une tentative de ..."
7,Sports,Les stations de ski prises en étau entre sauve...,"Les premières stations ont ouvert ce week-end,..."
8,Sports,Le difficile rôle de parent quand on a un enfa...,"Derrière chaque grand sportif, il y a souvent ..."
9,Sports,Le Mans renverse Marseille au Vélodrome et met...,Le promu sarthois se donne de l'air et sort de...


On peut aussi utiliser les Pipelines pour stocker les données récupérées
dans une base de données.

La prochaine partie du cours explique plus en détail le fonctionnement de Mongo, mais voici une bref introduction

Pour stocker les items dans des documents mongo :

In [41]:
import pymongo


class MongoPipeline(object):
    collection_name = 'scrapy_items'

    def open_spider(self, spider):
        self.client = pymongo.MongoClient()
        self.db = self.client["lemonde"]

    def close_spider(self, spider):
        self.client.close()

    def process_item(self, item, spider):
        self.db[self.collection_name].insert_one(dict(item))
        return item

Ici on redéfinit deux autres méthodes: `open_spider()`et
`close_spider()`, ces méthodes sont appelées comme leur nom l'indique,
lorsque la Spider est instanciée et fermée.

Ces méthodes nous permettent d'ouvrir la connexion Mongo et de la fermer
lorsque le scraping se termine. La méthode `process_item()` est appelé à
chaque fois qu'un item passe dans le mécanisme interne de scrapy. Ici,
la méthode permet d'insérer l'item en tant que document mongo.

Pour que ce pipeline soit appelé il faut l'ajouter dans les settings du
projet.

In [42]:
ITEM_PIPELINES = {
    'newscrawler.pipelines.TextPipeline': 100,
    'newscrawler.pipelines.MongoPipeline': 300
}

Le pipeline est ajoutée à la fin du process pour profiter des deux
précédents.

## Settings

Scrapy permet de gérer le comportement des spiders avec certains
paramètres. Comme expliqué dans le premier cours, il est important de
suivre des règles en respectant la structure des différents sites. Il
existe énormément de paramètres mais nous allons (dans le cadre de ce
cours) aborder les plus utilisés :

-   DOWNLOAD\_DELAY : Le temps de téléchargement entre chaque requête
    sur le même domaine ;
-   CONCURRENT\_REQUESTS\_PER\_DOMAIN : Nombre de requêtes simultanées
    par domaine ;
-   CONCURRENT\_REQUESTS\_PER\_IP : Nombre de requêtes simultanées par
    IP ;
-   DEFAULT\_REQUEST\_HEADERS : Headers HTTP utilisés pour les requêtes
    ;
-   ROBOTSTXT\_OBEY : Scrapy récupère le robots.txt et adapte le
    scraping en fonction des règles trouvées ;
-   USER\_AGENT : UserAgent utilisé pour faire les requêtes ;
-   BOT\_NAME : Nom du bot annoncé lors des requêtes
-   HTTPCACHE\_ENABLED : Utilisation du cache HTTP, utile lors du
    parcours multiple de la même page.

Le fichiers `settings.py` permet de définir les paramètres globaux d'un
projet. Si votre projet contient un grand nombre de spiders, il peut
être intéressant d'avoir des paramètres distincts pour chaque spider. Un
moyen simple est d'ajouter un attribut `custom_settings` à votre spider
:

In [43]:
class LeMondeSpider(scrapy.Spider):
    name = "lemonde"
    allowed_domains = ["lemonde.fr"]
    start_urls = ['http://lemonde.fr/']
    custom_settings = {
        "HTTPCACHE_ENABLED": True,
        "CONCURRENT_REQUESTS_PER_DOMAIN": 100
    }

C'est la fin de l'introduction à Scrapy. Vous pouvez passer à la suite `3Mongo`